# Counting Experiment Sensitivity — Solutions

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy  as np
import tables as tb
import pandas as pd
import matplotlib.pyplot as plt

import scipy.constants as constants
import scipy.stats     as stats
import scipy.optimize  as optimize

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, sys
from pathlib import Path

# Find the project root: honours FANAL_ROOT env-var, otherwise walks up from cwd
_env = os.environ.get('FANAL_ROOT') or os.environ.get('USCFANALDIR')
if _env and Path(_env, 'data').is_dir():
    rootpath = str(Path(_env).resolve())
else:
    rootpath = str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'data').is_dir() and (p / 'ana').is_dir()))
if rootpath not in sys.path:
    sys.path.insert(0, rootpath)
print('Fanal root : ', rootpath)

In [ ]:
import core.pltext  as pltext   # extensions for plotting histograms
import core.hfit    as hfit     # extension to fit histograms
import core.efit    as efit     # Fit Utilites - Includes Extend Likelihood Fit with composite PDFs
import core.confint as confint  # Confidence Intervals 
import core.utils   as ut       # generic utilities
import ana.fanal    as fn       # analysis functions specific to fanal
import     collpars as collpars # collaboration specific parameters
pltext.style()

In [ ]:
collaboration = collpars.collaboration
sel_ntracks   = collpars.sel_ntracks
sel_eblob2    = collpars.sel_eblob2
sel_erange    = collpars.sel_erange
sel_eroi      = collpars.sel_eroi

n_Bi_E        = collpars.n_Bi_E
n_Tl_E        = collpars.n_Tl_E
n_Bi_RoI      = collpars.n_Bi_RoI
n_Tl_RoI      = collpars.n_Tl_RoI


print('Collaboration             : {:s}'.format(collaboration))

In [ ]:
### data

# set the path to the data directory and filenames
dirpath = rootpath+'/data/'
filename = 'fanal_' + collaboration + '.h5'
print('Data path and filename : ', dirpath + filename)

# access the simulated data (DataFrames) for the different samples (Bi, Tl, bb) located in the data file
mcbi = pd.read_hdf(dirpath + filename, key = 'mc/bi214').fillna(0.)
mctl = pd.read_hdf(dirpath + filename, key = 'mc/tl208').fillna(0.)
mcbb = pd.read_hdf(dirpath + filename, key = 'mc/bb0nu').fillna(0.)

# set the names of the samples
# set the names of the samples
mc_samples         = [mcbi, mctl, mcbb] # list of the mc DFs
sample_names       = ['Bi', 'Tl', 'bb']
sample_names_latex = [ r'$^{214}$Bi', r'$^{208}$Tl', r'$\beta\beta0\nu$',] # str names of the mc samples

for i, mc in enumerate(mc_samples):
    print('MC Sample {:s}, number of simulated events = {:d}'.format(sample_names[i], len(mc)))

In [ ]:
nbkg   = n_Bi_RoI + n_Tl_RoI
nbkg

## Feldman-Cousins CI, half-life function, and plots

In [ ]:
nbkg   = n_Bi_RoI + n_Tl_RoI
n0     = int(nbkg)      # number of observed events (int)
cls    = (0.68, 0.90, 0.95)    # range of CL to show
factor = 1.5
nbins  = 100
nus    = np.linspace(0., factor * nbkg, 100) # range of number of signal events

# get the FC-CI functions for 68% and 95% CL
fcs    = [confint.get_fc_confinterval(nus, nbkg, cl = cl) for cl in cls]

for fc, cl in zip(fcs, cls):
    ci = fc(n0)
    print('FC Cover Interval, observed n = {:d}, at {:2.0f} % CL = ({:4.2f}, {:4.2f})'.format(n0, 100*cl, *ci))

In [ ]:
# plot the conficence band and the segment associated to an observation
pltext.canvas(1, 1, 6, 8)
ns   = np.arange(0, 2 * nbkg)
for fc, cl in zip(fcs, cls):    
    ys   = fc(ns)
    plt.fill_between(ns, *ys, alpha = 0.5, color = 'y', label = 'FC CI {:6.0f} % CL'.format(100*cl))
plt.plot(n0 * np.ones(len(nus)), nus, label = 'observed')
plt.legend(); plt.grid(which = 'both');
plt.title('FC CI bkg = {:4.2f}'.format(nbkg));
plt.xlabel('number of events'); plt.ylabel('number of signal events');
plt.tight_layout();

In [ ]:
def tau(nbb):
    eff      = collpars.eff_bb_RoI
    exposure = collpars.exposure
    tau      = nbb/collpars.exposure # implement here the formula of the lifetime
    return tau

In [ ]:
for fc, cl in zip(fcs, cls):
    ci   = fc(n0)
    taus = [tau(x) for x in ci]
    taus.reverse()
    print('FC Cover Interval, observed n = {:d}, at {:2.0f} % CL = ({:4.2e}, {:4.2e})'.format(n0, 100*cl, *taus))

In [ ]:
pltext.canvas(1, 1, 6, 8)
for fc, cl in zip(fcs, cls):    
    ys     = fc(ns)
    taus   = tau(ys)
    taus[taus == np.inf] = tau(nus[1])
    plt.fill_between(ns, *taus, 
                     alpha = 0.5, color = 'y', label = 'FC CI {:6.0f} % CL'.format(100*cl))
plt.plot(n0 * np.ones(len(nus)), tau(nus), label = 'observed')
plt.legend(); plt.grid(which = 'both');
plt.title('FC CI bkg = {:4.2f}'.format(nbkg));
plt.xlabel('number of events', fontsize = 14); plt.ylabel(r'$T^{0\nu}_{\beta\beta}$ (y)', fontsize = 14);
plt.yscale('log');
#plt.tight_layout();